# Links

1. https://youtu.be/JeAoWQTKTMw?si=4oiUnFohkYeJf7d6
2. https://youtu.be/GK16UzesqKs?si=Q4IhDmzTUR06hXNG
3. https://youtu.be/Sp3XfltBj3o?si=GoNp2MnP0ocHyFK5
4. https://youtu.be/Gu3U7l8G1J0?si=dTC50U3muuQvueUo
5. https://youtu.be/TqU4edl0-AM?si=132LPUR54uwAeJ__
6. https://youtu.be/X4cjpRkjqOg?si=XFgWYFyr0EHEU-He

https://youtu.be/qg_h7fs2HA0?si=Q7DG7L8-l0RJwLES

https://youtu.be/jLMJmljxKOw?si=hAdMk3FvO15ukEE6

In [ ]:
!pip install -U transformers peft accelerate torch librosa

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/ASR-whisper-finetuning/Testing_Audios

/content/drive/MyDrive/ASR-whisper-finetuning/Testing_Audios


# **kingabzpro**

## whisper-**large-v3**-urdu

In [ ]:
from transformers import pipeline

transcriber = pipeline("automatic-speech-recognition", model="kingabzpro/whisper-large-v3-urdu")

transcriber.model.generation_config.forced_decoder_ids = None
transcriber.model.generation_config.language = "ur"

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [ ]:
import os

for audio in os.listdir():
    print(f"Playback : {audio}")
    print(transcriber(audio)['text'])
    print("\n")

Playback : UAE_oil_prices.wav
یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کیلئے تیل کی قیمتوں میں اضافے کا اعلان کر دیا ہے فیول پرائس کمیٹی نے پیٹرل کی قیمتوں میں صفر آشاریہ دو سات درہم فی لیٹر تک اضافہ کر دیا پیٹرل فی لیٹر اس کی قیمت تین آشاریہ صفر پانچ درہم مقرر کر دی گئی ہے یہ پاکستانی کرنسی میں دو سو سولہ روپے ستاسی پیسے ہے ڈیجل کی فی لیٹر قیمت تین آشاریہ تین آٹھ درہم مقرر کی گئی جو پاکستانی کرنسی کا مطابق دو سو چالیس روپے تینتیس پیسے بنتی ہے


Playback : Prices_of_petrol__diesel.wav
پیٹرل اور ڈیزل کی قیمتوں میں کتنی کمی ہونے جا رہی ہے؟آئی ایم ایف کا پروگرام کب بحال ہوگا؟بشلی اور گیس کی قیمتوں میں اضافے کا کیا ہوگا؟ روپیہ اب گرے گا یا مزید اوپر جائے گا؟ دیکھئے وزیر خزانہ مفتا اسماعیل کا انٹرویو رات دس بجے کی ہیڈ لائنز کے بعد جیو نیوز کے پروگرام آج شاہزیب خانزادہ کے ساتھ میں


Playback : Iran_Attacks.wav
امریقی صدر ڈانلڈ ٹرمپ نے کہا ہے کہ ایران نے آج سخت حملوں کا اعلان کیا ہے بہتر ہے کہ وہ ایسا نہ کرے ایران نے اگر ایسا کیا تو انہیں ایسی طاقت سے نشانہ بنائیں گے جو پہلے کبھی نہیں د

# Model trained **(LoRA Finetuned)** on **Google Fleurs URDU** Language

### 1. **Kingabz --> 30% Degraded Dataset --> Large-v3-urdu**

```
Learning Rate = 5e-5
```

| Epoch | Train Loss | Val Loss | WER    | Epoch Time |
| ----- | ---------- | -------- | ------ | ---------- |
| 1     | 0.3648     | 0.3049   | 0.5755 | 4h 42m     |
| 2     | 0.2575     | 0.2982   | 0.7630 | 6h 36m     |
| 3     | 0.2125     | 0.3063   | 0.7106 | 6h 44m     |

**Observation:**

* Training loss decreases across epochs but the WER increases, indicating overfitting.
* Training time increases in later epochs due to model complexity and decoding overhead.
* Early stopping was applied at Epoch 4 due to high computational time and potential overfitting.



### Interpretation of Results

* The model demonstrates fast learning under 30% degraded speech, with training loss decreasing steadily across epochs.
* Although training loss improves, WER increases after Epoch 1, indicating weaker generalization.
* The gap between validation loss and WER reflects a mismatch between loss optimization and evaluation metrics.
* The best performance is achieved at **Epoch 1**, suggesting early convergence.
* Continued training may lead to overfitting under degraded conditions.
* Greedy decoding restricts sequence-level optimization and may increase WER.

In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from transformers import pipeline

# Path to your LoRA folder
lora_path = r"/content/drive/MyDrive/ASR-whisper-finetuning/kingabz_large_v3_30Downgrade"

# Load base Whisper model (VERY IMPORTANT: must match training base model)
# openai/whisper-large-v3
base_model_id = "kingabzpro/whisper-large-v3-urdu"

processor = WhisperProcessor.from_pretrained(base_model_id)
model = WhisperForConditionalGeneration.from_pretrained(base_model_id)

# Attach LoRA adapter
model = PeftModel.from_pretrained(model, lora_path)

model.eval()

# Create ASR pipeline
asr = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0 if torch.cuda.is_available() else -1
)

# Force language if needed
model.generation_config.language = "ur"
model.generation_config.forced_decoder_ids = None

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [ ]:
import os

for audio in os.listdir():
    print(f"Playback : {audio}")
    print(asr(audio)['text'])
    print("\n")

Playback : UAE_oil_prices.wav


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


یہا آپ کو بتائیں کہ متحد عرب امرات نے فوری کے لیے تیل کیمممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممممم لیلللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللللل


Playback : Prices_of_petrol__diesel.wav
پیٹرل اور ڈیزل کے قیمتوں میں کتنی کمی ہونے جا رہی ہیں آئی ایمیف کا پروگرم ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک ک 

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہو گئی ہے پی ٹی اے ذرائی کا کہنا ہے کہ سماجی رابیٹ کی ویب سائیٹ ٹوٹر بھی بححل ہو گئی ہے پی ٹی اے کارکنوں کے پرت شد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی


Playback : Big_News.wav
فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہو گئی ہے پی ٹی اے ذرائی کا کہنا ہے کہ سماجی رابیٹ کی ویب سائیٹ ٹوٹر بھی بححل ہو گئی ہے پی ٹی اے کارکنوں کے پرت شد مظاہروں کے بعد سوشل میڈیا سروس موطل کی گئی تھی


Playback : Telangana_News.mp3
وہی تلنگانہ میں اگلے تین دنوں ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت ت 

### 2. **Kingabz --> 30% Degraded Dataset --> Large-v3-urdu**

```
Learning Rate = 1e-5
```

| Epoch | Train Loss | Val Loss | WER    | Epoch Time |
| ----- | ---------- | -------- | ------ | ---------- |
| 1     | 0.5428     | 0.3755   | 0.2196 | 8m 29s     |
| 2     | 0.3369     | 0.3292   | 0.2105 | 8m 22s     |
| 3     | 0.3121     | 0.3184   | 0.2349 | 8m 12s     |
| 4     | 0.2863     | 0.3040   | 2.6805 | 8m 13s     |
| 5     | 0.2694     | 0.3046   | 1.6252 | 8m 15s     |

Best Model -- Epoch 2

**Observation:**

* Training and validation loss decrease gradually, showing stable learning.
* The lowest WER is achieved at Epoch 2, indicating optimal generalization.
* Sharp WER increases at later epochs suggest possible overfitting and decoding instability.


In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from transformers import pipeline

# Path to your LoRA folder
lora_path = r"/content/drive/MyDrive/ASR-whisper-finetuning/kingabz_large_v3_30Downgrade_LR"

# Load base Whisper model (VERY IMPORTANT: must match training base model)
# openai/whisper-large-v3
base_model_id = "kingabzpro/whisper-large-v3-urdu"

processor = WhisperProcessor.from_pretrained(base_model_id)
model = WhisperForConditionalGeneration.from_pretrained(base_model_id)

# Attach LoRA adapter
model = PeftModel.from_pretrained(model, lora_path)

model.eval()

# Create ASR pipeline
asr_30D = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0 if torch.cuda.is_available() else -1
)

# Force language if needed
model.generation_config.language = "ur"
model.generation_config.forced_decoder_ids = None

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [ ]:
import os

for audio in os.listdir():
    print(f"Playback : {audio}")
    print(asr_30D(audio)['text'])
    print("\n")

Playback : UAE_oil_prices.wav


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کے لیے تیل کی قیمتوں میں اضافہ کا اعلان کر دیا ہے فیول پرائس کمیٹی نے پیٹرل کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کر دیا ہے پیٹرل فی لیٹر اس کی قیمت 3.05 درہم مقرر کر دی گئی ہے یہ پاکستانی کرنسی میں 216 روپے 87 پیسے ہے ڈیجل کی فی لیٹر قیمت 3.38 درہم مقرر کی گئی جو پاکستانی کرنسی کا مطابق 240 روپے 33 پیسے بنتی ہے


Playback : Prices_of_petrol__diesel.wav
پیٹرل اور ڈیزل کی قیمتوں میں کتنی کمی ہونے جا رہی ہے آئی ایم ایف کا پروگرام کب بحال ہوگا بشلی اور گیس کی قیمتوں میں اضافہ کا کیا ہوگا روپیہ اب گرے گا یا مزید اوپر جائے گا دیکھئے وزیر خزانہ مفتا اسماعیل کا انٹرویو رات دس بجے کی ہیڈ لائنز کے بعد جیو نیوز کے پروگرام آج شاہزیب خانزادہ کے ساتھ میں


Playback : Iran_Attacks.wav
امریقی صدر ڈانلڈ ٹرمپ نے کہا ہے کہ ایران نے آج سخت حملوں کا اعلان کیا ہے بہتر ہے کہ وہ ایسا نہ کرے ایران نے اگر ایسا کیا تو انہیں ایسی طاقت سے نشانہ بنائیں گے جو پہلے کبھی نہیں دیکھی ہوگی


Playback : UAE_oil_prices.mp3
یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کے 

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذرائع کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس معتل کی گئی تھی


Playback : Telangana_News.mp3
وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش قیاسی کر دی گئی ہے محکمہ موسمیات نے اگلے تین دنوں تک تلنگانہ کے مختلف علاقوں میں بارش ہونے کا امکان ظاہر کیا ہے محکمہ موسمیات کے عدداروں کے مطابق شہر حیدرباد میں اگلے چار دنوں تک بارش ہو سکتی ہے کرونا کے بھرتے اثرات اور غیر موسمی بارش کو لے کر احتیاط برتنے کی بھی عام لوگوں سے لگتار اپیل کی جا رہی ہے


Playback : Telangana_News.wav
وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش قیاسی کر دی گئی ہے محکمہ موسمیات نے اگلے تین دنوں تک تلنگانہ کے مختلف علاقوں میں بارش ہونے کا امکان ظاہر کیا ہے محکمہ موسمیات کے عدداروں کے مطابق شہر حیدرباد میں اگلے چار دنوں تک بارش ہو سکتی ہے کرونا کے بھرتے اثرات اور غیر موسمی بارش کو لے کر احتیاط برتنے کی بھی عام لوگوں سے لگتار اپیل کی جا رہی ہے




### 3. **Kingabz --> 0% Degraded Dataset --> Large-v3-urdu**

```
Learning Rate = 1e-5
```

| Epoch | Train Loss | Val Loss | WER    | Epoch Time |
| ----- | ---------- | -------- | ------ | ---------- |
| 1     | 0.5432     | 0.3807   | 0.2206 | 6m 44s     |
| 2     | 0.3358     | 0.3289   | 0.2106 | 6m 40s     |
| 3     | 0.3097     | 0.3175   | 0.2377 | 6m 40s     |
| 4     | 0.2818     | 0.3054   | 2.6276 | 6m 42s     |
| 5     | 0.2674     | 0.3035   | 2.3400 | 6m 38s     |

**Observation:**

* Loss values decrease steadily, indicating stable training convergence.
* The best WER is achieved at Epoch 2.
* Significant WER spikes at later epochs suggest overfitting and decoding instability.



In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from transformers import pipeline

# Path to your LoRA folder
lora_path = r"/content/drive/MyDrive/ASR-whisper-finetuning/kingabz_large_v3_0Downgrade_LR"

# Load base Whisper model (VERY IMPORTANT: must match training base model)
# openai/whisper-large-v3
base_model_id = "kingabzpro/whisper-large-v3-urdu"

processor = WhisperProcessor.from_pretrained(base_model_id)
model = WhisperForConditionalGeneration.from_pretrained(base_model_id)

# Attach LoRA adapter
model = PeftModel.from_pretrained(model, lora_path)

model.eval()

# Create ASR pipeline
asr_0D = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0 if torch.cuda.is_available() else -1
)

# Force language if needed
model.generation_config.language = "ur"
model.generation_config.forced_decoder_ids = None

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [ ]:
import os

for audio in os.listdir():
    print(f"Playback : {audio}")
    print(asr_0D(audio)['text'])
    print("\n")

Playback : UAE_oil_prices.wav


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کے لیے تیل کی قیمتوں میں اضافہ کا اعلان کر دیا ہے فیول پرائس کمیٹی نے پیٹرل کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کر دیا ہے پیٹرل فی لیٹر اس کی قیمت 3.05 درہم مقرر کر دی گئی ہے یہ پاکستانی کرنسی میں 216 روپے 87 پیسے ہے ڈیجل کی فی لیٹر قیمت 3.38 درہم مقرر کی گئی جو پاکستانی کرنسی کا مطابق 240 روپے 33 پیسے بنتی ہے


Playback : Prices_of_petrol__diesel.wav
پیٹرل اور ڈیزل کی قیمتوں میں کتنی کمی ہونے جا رہی ہے آئی ایم ایف کا پروگرام کب بحال ہوگا بشلی اور گیس کی قیمتوں میں اضافہ کا کیا ہوگا روپیہ اب گرے گا یا مزید اوپر جائے گا دیکھئے وزیر خزانہ مفتا اسماعیل کا انٹرویو رات دس بجے کی ہیڈ لائنز کے بعد جیو نیوز کے پروگرام آج شاہزیب خانزادہ کے ساتھ میں


Playback : Iran_Attacks.wav
امریقی صدر ڈانلڈ ٹرمپ نے کہا ہے کہ ایران نے آج سخت حملوں کا اعلان کیا ہے بہتر ہے کہ وہ ایسا نہ کرے ایران نے اگر ایسا کیا تو انہیں ایسی طاقت سے نشانہ بنائیں گے جو پہلے کبھی نہیں دیکھی ہوگی


Playback : UAE_oil_prices.mp3
یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کے 

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس معتل کی گئی تھی


Playback : Telangana_News.mp3
وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش قیاسی کر دی گئی ہے محکمہ موسمیات نے اگلے تین دنوں تک تلنگانہ کے مختلف علاقوں میں بارش ہونے کا امکان ظاہر کیا ہے محکمہ موسمیات کے عدداروں کے مطابق شہر حیدرباد میں اگلے چار دنوں تک بارش ہو سکتی ہے کرونا کے بھرتے اثرات اور غیر موسمی بارش کو لے کر احتیاط بھرتنے کی بھی عام لوگوں سے لگتار اپیل کی جا رہی ہے


Playback : Telangana_News.wav
وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش قیاسی کر دی گئی ہے محکمہ موسمیات نے اگلے تین دنوں تک تلنگانہ کے مختلف علاقوں میں بارش ہونے کا امکان ظاہر کیا ہے محکمہ موسمیات کے عدداروں کے مطابق شہر حیدرباد میں اگلے چار دنوں تک بارش ہو سکتی ہے کرونا کے بھرتے اثرات اور غیر موسمی بارش کو لے کر احتیاط بھرتنے کی بھی عام لوگوں سے لگتار اپیل کی جا رہی ہے




# **OPENAI**

## whisper-**large-v3**

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe_orig = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

In [ ]:
import os

for i, audio in enumerate(os.listdir(), start=1):
    print(f"{i}. Playback : {audio}")
    print(pipe_orig(audio,generate_kwargs={"language": "urdu"})['text'])
    print("\n")

1. Playback : UAE_oil_prices.wav
 یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کے لیے تیل کی قیمتوں میں اضافے کا اعلان کر دیا ہے فیول پرائس کمیٹی نے پیٹرل کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کر دیا پیٹرل فی لیٹر اس کی قیمت 3.05 درہم مقرر کر دی گئی ہے یہ پاکستانی کرنسی میں 216 روپے 87 پیسے ہے ڈیجل کی فی لیٹر قیمت 3.38 درہم مقرر کی گئی جو پاکستانی کرنسی کا مطابق 240 روپے 33 پیسے بنتی ہے


2. Playback : Prices_of_petrol__diesel.wav
 پیٹرل اور ڈیزل کی قیمتوں میں کتنی کمی ہونے جا رہی ہے؟ آئی ایم ایف کا پروگرام کب بحال ہوگا؟ بشلی اور گیس کی قیمتوں میں اضافے کا کیا ہوگا؟ روپیہ اب گرے گا یا مزید اوپر جائے گا؟ دیکھئے وزیر خزانہ مفتا اسماعیل کا انٹرویو رات دس بجے کی ہیڈ لائنز کے بعد جیو نیوز کے پروگرام آج شہزیب خانزادہ کے ساتھ میں


3. Playback : Iran_Attacks.wav
 امریکی صدر ڈانلڈ ٹرمپ نے کہا ہے کہ ایران نے آج سخت حملوں کا اعلان کیا ہے بہتر ہے کہ وہ ایسا نہ کرے ایران نے اگر ایسا کیا تو انہیں ایسی طاقت سے نشانہ بنائیں گے جو پہلے کبھی نہیں دیکھی ہوگی


4. Playback : UAE_oil_prices.mp3
 یہاں

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


 فیس بک اور یوٹیوب کی سرویس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذرائقہ کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سرویس موتل کی گئی تھی


10. Playback : Big_News.wav
 فیس بک اور یوٹیوب کی سرویس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذرائقہ کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سرویس موتل کی گئی تھی


11. Playback : Telangana_News.mp3
 وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش قیاسی کر دی گئی ہے محکمہ موسمیات نے اگلے تین دن تک تلنگانہ کے مختلف علاقوں میں بارش ہونے کا امکان ظاہر کیا ہے محکمہ موسمیات کے عدداروں کے مطابق شہر حیدرباد میں اگلے چار دنوں تک بارش ہو سکتی ہے کرونا کے برتے اثرات اور غیر موسمی بارش کو لے کر احتیاط برتنے کی بھی عام لوگوں سے لگتار اپیل کی جا رہی ہے


12. Playback : Telangana_News.wav
 وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش قیاسی کر دی گئی ہے محکمہ موسمیات نے اگلے تین دن تک تلنگا

# Model trained **(LoRA Finetuned)** on **Google Fleurs URDU** Language

### 1. **OpenAI --> 0% Degraded Dataset --> Large-v3-urdu**

```
Learning Rate = 1e-5
```

| Epoch | Train Loss | Val Loss | WER    | Epoch Time |
| ----- | ---------- | -------- | ------ | ---------- |
| 1     | 0.5899     | 0.4581   | 0.3533 | 6m 38s     |
| 2     | 0.3723     | 0.3480   | 0.2275 | 6m 40s     |
| 3     | 0.3205     | 0.3288   | 2.6998 | 6m 37s     |
| 4     | 0.2948     | 0.3168   | 3.1201 | 6m 37s     |
| 5     | 0.2764     | 0.3140   | 2.9795 | 6m 38s     |

**Observation:**

* Training and validation loss decrease steadily across epochs → model is learning stably.
* WER improves significantly until **Epoch 2 (best: 0.2275)**.
* After Epoch 2, WER increases drastically (2.6–3.1) despite decreasing validation loss.
* This mismatch suggests a **decoding/evaluation issue**, not normal overfitting.
* **Best checkpoint: Epoch 2.**


In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from peft import PeftModel

# Path to your LoRA adapter folder
lora_path = r"/content/drive/MyDrive/ASR-whisper-finetuning/openai_large_v3_0Downgrade_LR"

# Base model (must match training base)
base_model_id = "openai/whisper-large-v3"

# 1️⃣ Load base model
base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 2️⃣ Load LoRA adapter weights
model = PeftModel.from_pretrained(base_model, lora_path)

# 3️⃣ Load processor (tokenizer + feature extractor)
processor = AutoProcessor.from_pretrained(lora_path)

# 4️⃣ Create ASR pipeline
pipe_lora = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [ ]:
import os

for i, audio in enumerate(os.listdir(), start=1):
    print(f"{i}. Playback : {audio}")
    print(pipe_lora(audio,generate_kwargs={"language": "urdu"})['text'])
    print("\n")

1. Playback : UAE_oil_prices.wav


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کے لیے تیل کی قیمتوں میں اضافہ کا اعلان کر دیا ہے فیول پرائس کمیٹی نے پیٹرل کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کر دیا ہے پیٹرل فی لیٹر اس کی قیمت 3.05 درہم مقرر کر دی گئی ہے یہ پاکستانی کرنسی میں 216 روپے 87 پیسے ہے ڈیجل کی فی لیٹر قیمت 3.38 درہم مقرر کی گئی جو پاکستانی کرنسی کا مطابق 240 روپے 33 پیسے بنتی ہے


2. Playback : Prices_of_petrol__diesel.wav
پیٹرل اور ڈیزل کی قیمتوں میں کتنی کمی ہونے جا رہی ہے آئی ایم ایف کا پروگرام کب بحال ہوگا بشلی اور گیس کی قیمتوں میں اضافہ کا کیا ہوگا روپیہ اب گرے گا یا مزید اوپر جائے گا دیکھئے وزیر خزانہ مفتا اسماعیل کا انٹرویو رات دس بجے کی ہیڈ لائنز کے بعد جیو نیوز کے پروگرام آج شاہزیب خانزادہ کے ساتھ میں


3. Playback : Iran_Attacks.wav
امریکی صدر ڈانلڈ ٹرمپ نے کہا ہے کہ ایران نے آج سخت حمل کا اعلان کیا ہے بہتر ہے کہ وہ ایسا نہ کرے ایران نے اگر ایسا کیا ہے تو انہیں ایسی طاقت سے نشانہ بنائیں گے جو پہلے کبھی نہیں دیکھی ہوگی


4. Playback : UAE_oil_prices.mp3
یہاں آپ کو بتائیں گے متحدہ عرب امارات نے

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


فیس بک اور یوٹیوب کی سروس پاکستان میں بحال ہونا شروع ہو گئی ہے پی ٹی اے ذراعی کا کہنا ہے کہ سماجی رابطے کی ویب سائٹ ٹوٹر بھی بحال ہونا شروع ہو گئی ہے پی ٹی اے کارکنوں کے پرتشدد مظاہروں کے بعد سوشل میڈیا سروس معتل کی گئی تھی


11. Playback : Telangana_News.mp3
وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش قیاسی کر دی گئی ہے محکمہ موسمیات نے اگلے تین دنوں تک تلنگانہ کے مختلف علاقوں میں بارش ہونے کا امکان ظاہر کیا ہے محکمہ موسمیات کے عدداروں کے مطابق شہر حیدرباد میں اگلے چار دنوں تک بارش ہو سکتی ہے کرونا کے بھرتے اثرات اور غیر موسمی بارش کو لے کر احتیاط برتنے کی بھی عام لوگوں سے لگتار اپیل کی جا رہی ہے


12. Playback : Telangana_News.wav
وہیں تلنگانہ میں اگلے تین دنوں تک بارش کی پیش قیاسی کر دی گئی ہے محکمہ موسمیات نے اگلے تین دنوں تک تلنگانہ کے مختلف علاقوں میں بارش ہونے کا امکان ظاہر کیا ہے محکمہ موسمیات کے عدداروں کے مطابق شہر حیدرباد میں اگلے چار دنوں تک بارش ہو سکتی ہے کرونا کے بھرتے اثرات اور غیر موسمی بارش کو لے کر احتیاط بھرتنے کی بھی عام لوگوں سے لگتار اپیل کی جا رہی ہے




### 2. **OpenAI --> 30% Degraded Dataset --> Large-v3-urdu**

```
Learning Rate = 1e-5
```

| Epoch | Train Loss | Val Loss | WER    | Epoch Time |
| ----- | ---------- | -------- | ------ | ---------- |
| 1     | 0.5899     | 0.4581   | 0.3533 | 6m 38s     |
| 2     | 0.3723     | 0.3480   | 0.2275 | 6m 40s     |
| 3     | 0.3205     | 0.3288   | 2.6998 | 6m 37s     |
| 4     | 0.2948     | 0.3168   | 3.1201 | 6m 37s     |
| 5     | 0.2764     | 0.3140   | 2.9795 | 6m 38s     |

**Observation:**




In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from peft import PeftModel

# Path to your LoRA adapter folder
lora_path = r"/content/drive/MyDrive/ASR-whisper-finetuning/openai_large_v3_30Downgrade_LR"

# Base model (must match training base)
base_model_id = "openai/whisper-large-v3"

# 1️⃣ Load base model
base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 2️⃣ Load LoRA adapter weights
model = PeftModel.from_pretrained(base_model, lora_path)

# 3️⃣ Load processor (tokenizer + feature extractor)
processor = AutoProcessor.from_pretrained(lora_path)

# 4️⃣ Create ASR pipeline
pipe_lora = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
)

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [ ]:
import os

for i, audio in enumerate(os.listdir(), start=1):
    print(f"{i}. Playback : {audio}")
    print(pipe_lora(audio,generate_kwargs={"language": "urdu"})['text'])
    print("\n")

1. Playback : UAE_oil_prices.wav
یہاں آپ کو بتائیں گے متحدہ عرب امارات نے فروری کے لیے تیل کی قیمتوں میں اضافہ کا اعلان کر دیا ہے فیول پرائس کمیٹی نے پیٹرل کی قیمتوں میں 0.27 درہم فی لیٹر تک اضافہ کر دیا ہے پیٹرل فی لیٹر اس کی قیمت 3.05 درہم مقرر کر دی گئی ہے یہ پاکستانی کرنسی میں 216 روپے 87 پیسے ہے ڈیجل کی فی لیٹر قیمت 3.78 درہم مقرر کی گئی جو پاکستانی کرنسی کا مطابق 240 روپے 33 پیسے بنتی ہے


2. Playback : Prices_of_petrol__diesel.wav
پیٹرل اور ڈیزل کی قیمتوں میں کتنی کمی ہونے جا رہی ہے آئی ایم ایف کا پروگرام کب بحال ہوگا بشلی اور گیس کی قیمتوں میں اضافہ کا کیا ہوگا روپیہ اب گرے گا یا مزید اوپر جائے گا دیکھئے وزیر خزانہ مفتا اسماعیل کا انٹرویو رات دس بجے کی ہیڈ لائنز کے بعد جیو نیوز کے پروگرام آج شاہزیب خانزادہ کے ساتھ میں


3. Playback : Iran_Attacks.wav
امری کی صدر ڈانلڈ ٹرمپ نے کہا ہے کہ ایران نے آج سخت حملوں کا اعلان کیا ہے بہتر ہے کہ وہ ایسا نہ کرے ایران نے اگر ایسا کیا ہے تو انہیں ایسی طاقت سے نشانہ بنائیں گے جو پہلے کبھی نہیں دیکھی ہوگی


4. Playback : UAE_oil_prices.mp3
یہاں